In [128]:
import pyvista as pv
from pathlib import Path

In [129]:
sub = '50017L'
pose = 'flexion'
savepath = Path(f'../outputs/CA-35T/meshes/{sub}')
mesh_paths = list(savepath.glob(f'**/*_{pose}-*.3mf'))
mesh_paths

[PosixPath('../outputs/CA-35T/meshes/50017L/tpmPrint_flexion-35T.3mf')]

In [130]:
import trimesh
instron_tmesh = trimesh.load(mesh_paths[0])

print("Loaded geometry keys:", list(instron_tmesh.geometry.keys()))
instron_tmesh.show()

Loaded geometry keys: ['boneJig', 'cartilage']


# Proper checks to make sure bones are in the correct position

In [124]:
import pyvista as pv
from pathlib import Path
import numpy as np

from phd_helpers.paths import (
    get_subject_stl_path, get_bone_inertia, get_bone_transforms, get_relative_transform_new_basis, transform_mesh, pose2idCMC, get_mesh
)
from phd_helpers.experiments import get_sensor_loc, build_sensor_mesh

In [125]:
sub = '50017L'
savepath = Path(f'../outputs/CA-35T/meshes/{sub}')
mesh_paths = list(savepath.glob(f'**/*.vtp'))
mesh_paths

[PosixPath('../outputs/CA-35T/meshes/50017L/tpm_cartilage_pinch_load-35T.vtp'),
 PosixPath('../outputs/CA-35T/meshes/50017L/tpm_boneJig_abduction-35T.vtp'),
 PosixPath('../outputs/CA-35T/meshes/50017L/tpm_boneJig_adduction-35T.vtp'),
 PosixPath('../outputs/CA-35T/meshes/50017L/cart-tpm-35T_repaired.vtp'),
 PosixPath('../outputs/CA-35T/meshes/50017L/tpm_boneJig_flexion-35T.vtp'),
 PosixPath('../outputs/CA-35T/meshes/50017L/tpm_boneJig_pinch_load-35T.vtp'),
 PosixPath('../outputs/CA-35T/meshes/50017L/tpm_cartilage_adduction-35T.vtp'),
 PosixPath('../outputs/CA-35T/meshes/50017L/tpm_cartilage_flexion-35T.vtp'),
 PosixPath('../outputs/CA-35T/meshes/50017L/tpm_cartilage_abduction-35T.vtp'),
 PosixPath('../outputs/CA-35T/meshes/50017L/mc1_boneJig-35T.vtp'),
 PosixPath('../outputs/CA-35T/meshes/50017L/mc1_cartilage-35T.vtp'),
 PosixPath('../outputs/CA-35T/meshes/50017L/tpm_cartilage_extension-35T.vtp'),
 PosixPath('../outputs/CA-35T/meshes/50017L/tpm_boneJig_extension-35T.vtp')]

In [126]:
pose = 'extension'
subject, sideL = sub[:-1], sub[-1]
stl_path = get_subject_stl_path(subject, sideL)
mc1_centroid, _, mc1_axes = get_bone_inertia(stl_path, 'mc1')
try:
    R, t = get_relative_transform_new_basis(get_bone_transforms(pose2idCMC(pose), stl_path), 'tpm', 'mc1', mc1_centroid, mc1_axes)
except:
    R, t = np.eye(3), np.zeros(3)

tpm = get_mesh(stl_path, 'tpm')
mc1 = get_mesh(stl_path, 'mc1')
mc1 = transform_mesh(mc1, mc1_axes, mc1_centroid, inverse=True)
tpm = transform_mesh(tpm, mc1_axes, mc1_centroid, inverse=True)
tpm = transform_mesh(tpm, R, t)

mc1_jig = pv.read(list(savepath.glob(f'mc1_boneJig*.vtp'))[0])
tpm_jig = pv.read(list(savepath.glob(f'tpm_boneJig_{pose}*.vtp'))[0])

sensor_centre = get_sensor_loc(mc1_jig)
sensor = build_sensor_mesh(sensor_centre)

In [127]:
pl = pv.Plotter()

pl.add_mesh(tpm, color='magenta', style='wireframe')
pl.add_mesh(mc1, color='magenta', style='wireframe')

pl.add_mesh(tpm_jig, color='white', opacity=1)
pl.add_mesh(mc1_jig, color='white', opacity=1)

pl.add_mesh(sensor, color='green', opacity=0.8)

pl.show()

Widget(value='<iframe src="http://localhost:54160/index.html?ui=P_0x3b841e3c0_41&reconnect=auto" class="pyvist…

#### Check repaired region

In [111]:
sub = '50017L'
bone = 'tpm'
savepath = Path(f'../outputs/CA-35T/meshes/{sub}')
mesh_paths = list(savepath.glob(f'**/{bone}_cart*.vtp'))
mesh = pv.read(mesh_paths[0])

In [112]:
pl = pv.Plotter()
pl.add_mesh(mesh, scalars='repaired', cmap='Accent_r')
pl.show()

Widget(value='<iframe src="http://localhost:54160/index.html?ui=P_0x3b841d7c0_39&reconnect=auto" class="pyvist…

In [114]:
mesh.is_manifold

True